# Pipeline 6 : Regrouper les molecules en familles chimiques

La question du chercheur : parmi toutes ces molecules, y a-t-il des grandes familles, et certaines sont-elles plus prometteuses que d'autres ?

Jusqu'ici on a traite chaque molecule individuellement. Mais un chimiste raisonne souvent par familles, des groupes de molecules qui partagent un squelette commun. Le clustering va decouvrir ces familles automatiquement, sans qu'on lui dise lesquelles chercher. L'interet metier est direct : si une famille concentre beaucoup de molecules actives, elle merite d'etre exploree en priorite, en synthetisant des variantes autour de ce squelette gagnant.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import joblib

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"
PALETTE = [C_BLEU, C_ORANGE, C_VERT, C_ROUGE, "#5f0f40", "#d4a017", "#3a86ff"]

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('egfr_descripteurs.csv')
fingerprints = np.load('egfr_fingerprints.npy')
df['activite'] = np.where(df['pIC50'] >= 6, 'actif',
                          np.where(df['pIC50'] < 5, 'inactif', 'intermediaire'))
print(f"Molecules : {len(df)}")

# 1. Reduire les empreintes avant de clusteriser

Les empreintes ont 2048 dimensions, un espace dans lequel les notions de distance deviennent instables, ce qu'on appelle la maledicction de la dimension. On applique d'abord une PCA pour condenser l'information dans une trentaine de composantes, ce qui rend le clustering plus stable et plus rapide.

In [ ]:
pca_pre = PCA(n_components=30, random_state=RANDOM_STATE)
X_reduit = pca_pre.fit_transform(fingerprints)
var_expliquee = pca_pre.explained_variance_ratio_.sum() * 100
print(f"Les 30 composantes retiennent {var_expliquee:.0f}% de la variance des empreintes.")
print("On clusterise sur cet espace condense plutot que sur les 2048 bits bruts.")
gc.collect()

# 2. Choisir le nombre de familles : coude et silhouette

In [ ]:
inerties, silhouettes = [], []
ks = range(2, 10)
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_reduit)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_reduit, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(ks), inerties, marker='o', color=C_BLEU, linewidth=2)
axes[0].set_title("Methode du coude")
axes[0].set_xlabel("Nombre de familles k")
axes[0].set_ylabel("Inertie intra-cluster")

axes[1].plot(list(ks), silhouettes, marker='o', color=C_ORANGE, linewidth=2)
meilleur_k = list(ks)[int(np.argmax(silhouettes))]
axes[1].axvline(meilleur_k, color=C_VERT, linestyle='--', label=f'Meilleure silhouette : k = {meilleur_k}')
axes[1].set_title("Score de silhouette")
axes[1].set_xlabel("Nombre de familles k")
axes[1].set_ylabel("Silhouette")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Le coude indique le nombre de familles au-dela duquel ajouter des groupes n'apporte plus grand chose, et la silhouette mesure la nettete de la separation. On retient un nombre de familles qui combine une bonne silhouette et une interpretabilite chimique : trop de familles donnerait des micro-groupes impossibles a caracteriser, trop peu melangerait des chimies distinctes.")

In [ ]:
K = 5
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
df['famille'] = kmeans.fit_predict(X_reduit)
print(f"Familles chimiques : {K}")
print(df['famille'].value_counts().sort_index())

# 3. Le graphique signature : quelles familles sont les plus actives ?

Pour chaque famille, on regarde le pIC50 moyen et le taux de molecules actives. C'est l'information qui oriente directement le chercheur vers les familles a explorer.

In [ ]:
profil = df.groupby('famille').agg(
    taille=('pIC50', 'size'),
    pIC50_moyen=('pIC50', 'mean'),
    poids_moyen=('poids_moleculaire', 'mean'),
    logP_moyen=('logP', 'mean'),
    taux_actifs=('activite', lambda s: (s == 'actif').mean() * 100)
).round(2)
print(profil)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

bars = axes[0].bar(profil.index.astype(str), profil['pIC50_moyen'],
                   color=[PALETTE[i] for i in profil.index], edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"{bar.get_height():.1f}", ha='center', fontsize=10)
axes[0].axhline(6, color=C_ROUGE, linestyle='--', linewidth=1.2, label='Seuil actif')
axes[0].set_title("Puissance moyenne par famille chimique")
axes[0].set_xlabel("Famille")
axes[0].set_ylabel("pIC50 moyen")
axes[0].legend()

bars2 = axes[1].bar(profil.index.astype(str), profil['taux_actifs'],
                    color=[PALETTE[i] for i in profil.index], edgecolor='white')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.0f}%", ha='center', fontsize=10)
axes[1].set_title("Taux de molecules actives par famille")
axes[1].set_xlabel("Famille")
axes[1].set_ylabel("Pourcentage d'actives")

plt.tight_layout()
plt.show()

print("Voila l'outil de priorisation que le chercheur attend. Les familles avec le pIC50 moyen le plus eleve et le taux d'actives le plus fort sont les zones chaudes du projet. Concretement, si une famille affiche 70% d'actives, synthetiser de nouvelles molecules autour de ce squelette a de bonnes chances de donner encore des molecules actives. A l'inverse, les familles a faible taux d'actives sont probablement des impasses qu'on peut deprioriser.")

# 4. Comparaison avec DBSCAN

In [ ]:
voisins = NearestNeighbors(n_neighbors=5).fit(X_reduit)
distances, _ = voisins.kneighbors(X_reduit)
distances_triees = np.sort(distances[:, -1])

plt.figure(figsize=(11, 5))
plt.plot(distances_triees, color=C_BLEU, linewidth=1.8)
plt.title("Graphique des k-distances pour regler DBSCAN")
plt.xlabel("Molecules triees")
plt.ylabel("Distance au 5e voisin")
plt.tight_layout()
plt.show()

# On choisit eps au niveau du coude de la courbe
eps_choisi = np.percentile(distances_triees, 90)
dbscan = DBSCAN(eps=eps_choisi, min_samples=5)
labels_db = dbscan.fit_predict(X_reduit)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_bruit = (labels_db == -1).sum()

print(f"DBSCAN : {n_clusters_db} familles denses trouvees, {n_bruit} molecules classees comme atypiques ({n_bruit/len(df)*100:.0f}%)")
print()
print("DBSCAN raconte une histoire complementaire de KMeans. La ou KMeans range de force chaque molecule dans une famille, DBSCAN isole les coeurs denses et laisse de cote les molecules structurellement uniques. Ces molecules atypiques sont interessantes en soi : un squelette chimique que personne d'autre ne partage peut etre une piste originale, ou au contraire une erreur dans les donnees. Pour la segmentation exploitable du marche chimique, KMeans reste plus pratique car chaque molecule recoit une famille.")

# 5. Deploiement

In [ ]:
joblib.dump({'pca_pre': pca_pre, 'kmeans': kmeans},
            'modele_clustering_familles.pkl')
df[['canonical_smiles', 'pIC50', 'activite', 'famille']].to_csv('egfr_familles.csv', index=False)
print("Sauvegarde : modele_clustering_familles.pkl et egfr_familles.csv")

# Conclusion

Le clustering a fait emerger des familles chimiques aux profils d'activite nettement contrastes, ce qui transforme une liste plate de molecules en une carte de priorites pour l'equipe de recherche. Certaines familles concentrent les molecules actives et meritent une exploration approfondie, d'autres sont des impasses.

La limite a garder en tete est que ces familles sont definies par la similarite des empreintes, une notion statistique qui ne coincide pas toujours avec la notion de famille qu'aurait un chimiste, fondee sur le squelette moleculaire. C'est justement pour combler cet ecart qu'on analysera les scaffolds de Murcko dans le notebook bonus, une approche qui definit les familles sur le squelette reel des molecules.